In [1]:
import os
import torch

# The CSV containing both text and image references
CSV_PATH = '/content/drive/.shortcut-targets-by-id/1AhfnCMVwculYjFHcvMlYXWIEvtKo0i3y/document_classifier/cache/all_text_dataset.csv'

# Base directory 
IMG_BASE_DIR = '/content/drive/.shortcut-targets-by-id/1pCrT-i-OuLmydhrMGwT2JJ-n0AOaAKDQ/ColabNotebooks/document-classifier/data/scanned_docs'

# Target path for saving/loading models
SAVE_DIR = '/content/drive/.shortcut-targets-by-id/1pCrT-i-OuLmydhrMGwT2JJ-n0AOaAKDQ/ColabNotebooks/document-classifier/models/trained_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# PRE-TRAINED WEIGHT PATHS 
RESNET_WEIGHTS = os.path.join(SAVE_DIR, 'best_resnet18.pth')
BERT_CHECKPOINT = os.path.join(SAVE_DIR, 'bert_checkpoint.pth')

# HYPERPARAMETERS
MODEL_NAME = 'bert-base-uncased'  
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-5
MAX_LEN = 512

# Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Cell 1 Complete. Variables initialized. Using device: cuda


In [2]:
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoTokenizer

# LOAD DATA & ENCODE LABELS 
df = pd.read_csv(CSV_PATH)
df['text'] = df['text'].fillna('[NO_TEXT_DETECTED]').astype(str)

le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_idx']
)

# JOINT DATASET CLASS
class JointDocumentDataset(Dataset):
    def __init__(self, dataframe, img_dir, tokenizer, transform, max_len=512):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load Image (Match CNN size 128x128)
        img_path = os.path.join(self.img_dir, row['image_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (128, 128), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        # Tokenize Text (Match BERT requirements)
        encoding = self.tokenizer(
            str(row['text']),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'image': image,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(row['label_idx'], dtype=torch.long)
        }

# TRANSFORMS & LOADERS 
fusion_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = JointDocumentDataset(train_df, IMG_BASE_DIR, tokenizer, fusion_transform, MAX_LEN)
val_ds = JointDocumentDataset(val_df, IMG_BASE_DIR, tokenizer, fusion_transform, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Cell 2 Complete. Prepared 2785 samples with 10 classes.


In [6]:
import torch.nn as nn
from torchvision import models
from transformers import AutoModel

class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes, bert_model_name):
        super(MultimodalFusionModel, self).__init__()
        
        # 1. IMAGE BRANCH (ResNet18)
        self.resnet = models.resnet18(weights=None)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        self.resnet_features = nn.Sequential(*list(self.resnet.children())[:-1])
        
        # 2. TEXT BRANCH (BERT)
        self.bert = AutoModel.from_pretrained(bert_model_name)
        
        # 3. FUSION CLASSIFIER
        self.classifier_head = nn.Sequential(
            nn.Linear(512 + 768, 256),
            nn.BatchNorm1d(256), 
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        img_feats = torch.flatten(self.resnet_features(image), 1)
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = bert_outputs.pooler_output
        
        combined_feats = torch.cat((img_feats, text_feats), dim=1)
        return self.classifier_head(combined_feats)


model = MultimodalFusionModel(num_classes, MODEL_NAME).to(device)

# Points  to the file i saved with the emergency code so not to start from scratch after a training failed after reaching 99% for epoch 1
RECOVERED_PATH = os.path.join(SAVE_DIR, 'recovered_epoch_1.pth')

if os.path.exists(RECOVERED_PATH):
    print(f"Restoring weights from partial training: {RECOVERED_PATH}")
    model.load_state_dict(torch.load(RECOVERED_PATH, map_location=device))
else:
    print("Recovered file not found. Starting from fresh branch weights.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Restoring weights from partial training: /content/drive/.shortcut-targets-by-id/1pCrT-i-OuLmydhrMGwT2JJ-n0AOaAKDQ/ColabNotebooks/document-classifier/models/trained_models/recovered_epoch_1.pth

Cell 3 Complete. Model is back in memory.


In [7]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import os

#  OPTIMIZER, LOSS, and SCHEDULER 
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


best_val_acc = 0.0
early_stop_patience = 4
counter = 0
fusion_checkpoint_path = os.path.join(SAVE_DIR, 'fusion_checkpoint.pth')
best_fusion_path = os.path.join(SAVE_DIR, 'best_fusion_model.pth')

print(f"Starting Fusion Training...")

# TRAINING LOOP 
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct_train, total_train = 0.0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        if images.size(0) < 2:
            model.eval() 
            with torch.no_grad():
                outputs = model(images, input_ids, attention_mask)
            model.train() 
            continue 
        
        optimizer.zero_grad()
        outputs = model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # Validation Phase
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct_val / total_val
    
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}: Val Acc: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr}")

    # SAVE CHECKPOINT (For Resuming)
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': best_val_acc,
    }
    torch.save(checkpoint, fusion_checkpoint_path)

    # SAVE BEST MODEL (For Inference) 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_fusion_path)
        print(f"New Best Fusion Model Saved (Acc: {val_acc:.4f})")
        counter = 0
    else:
        counter += 1
        if counter >= early_stop_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


Starting Fusion Training...


Epoch 1/10:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch 1: Val Acc: 0.8666 | Val Loss: 0.6970 | LR: 1e-05
⭐️ New Best Fusion Model Saved (Acc: 0.8666)


Epoch 2/10:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch 2: Val Acc: 0.9024 | Val Loss: 0.5274 | LR: 1e-05
⭐️ New Best Fusion Model Saved (Acc: 0.9024)


...


Epoch 6: Val Acc: 0.9139 | Val Loss: 0.3641 | LR: 1e-05


Epoch 7/10:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch 7: Val Acc: 0.9125 | Val Loss: 0.3367 | LR: 1e-05


Epoch 8/10:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch 8: Val Acc: 0.9053 | Val Loss: 0.3428 | LR: 1e-05
Early stopping at epoch 8

Final Training Complete!
